# Chapter 2: Stochastic Optimization

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/your-repo/blob/main/notebooks/ch02_stochastic_optimization.ipynb)

## Learning Objectives

By the end of this notebook, you will be able to:
1. Implement Stochastic Gradient Descent (SGD) from scratch
2. Understand the effects of mini-batch size on optimization
3. Visualize gradient variance and variance reduction techniques
4. Compare full-batch GD vs SGD convergence behavior
5. Implement and compare learning rate scheduling strategies

---


In [ ]:
# Setup and imports
import numpy as np
import matplotlib.pyplot as plt
from matplotlib import animation
from matplotlib.patches import FancyArrowPatch
import warnings
warnings.filterwarnings('ignore')

# Try to import ipywidgets
try:
    from ipywidgets import interact, interactive, FloatSlider, IntSlider, Dropdown
    WIDGETS_AVAILABLE = True
except ImportError:
    WIDGETS_AVAILABLE = False
    print("ipywidgets not available. Install with: pip install ipywidgets")

# For Colab animation display
from IPython.display import HTML

np.random.seed(42)
print("Setup complete!")


## 1. SGD Implementation from Scratch

In machine learning, we often minimize empirical risk:
$$\min_w \frac{1}{n} \sum_{i=1}^n \ell(w; x_i, y_i)$$

**Full-batch Gradient Descent** uses all data points:
$$w_{t+1} = w_t - \alpha \frac{1}{n} \sum_{i=1}^n \nabla \ell(w_t; x_i, y_i)$$

**Stochastic Gradient Descent** uses a random subset (or single sample):
$$w_{t+1} = w_t - \alpha \nabla \ell(w_t; x_{i_t}, y_{i_t})$$

where $i_t$ is randomly chosen at each step.


In [ ]:
class LinearRegressionSGD:
    """
    Linear Regression with various gradient descent methods.

    Objective: min_w (1/n) * ||Xw - y||^2
    """

    def __init__(self, X, y):
        """
        Parameters:
        -----------
        X : ndarray of shape (n_samples, n_features)
        y : ndarray of shape (n_samples,)
        """
        self.X = X
        self.y = y
        self.n_samples, self.n_features = X.shape

    def loss(self, w):
        """Compute mean squared error loss."""
        residuals = self.X @ w - self.y
        return np.mean(residuals ** 2)

    def full_gradient(self, w):
        """Compute full-batch gradient."""
        residuals = self.X @ w - self.y
        return (2 / self.n_samples) * self.X.T @ residuals

    def stochastic_gradient(self, w, batch_indices):
        """Compute stochastic gradient for given batch."""
        X_batch = self.X[batch_indices]
        y_batch = self.y[batch_indices]
        residuals = X_batch @ w - y_batch
        return (2 / len(batch_indices)) * X_batch.T @ residuals

    def gd(self, w0, lr=0.01, max_iter=1000):
        """Full-batch Gradient Descent."""
        w = w0.copy()
        history = {'w': [w.copy()], 'loss': [self.loss(w)], 'grad_norm': []}

        for t in range(max_iter):
            grad = self.full_gradient(w)
            history['grad_norm'].append(np.linalg.norm(grad))
            w = w - lr * grad
            history['w'].append(w.copy())
            history['loss'].append(self.loss(w))

        return w, history

    def sgd(self, w0, lr=0.01, max_iter=1000, batch_size=1):
        """Stochastic Gradient Descent with mini-batches."""
        w = w0.copy()
        history = {'w': [w.copy()], 'loss': [self.loss(w)],
                   'grad_norm': [], 'stoch_grad_norm': []}

        for t in range(max_iter):
            # Sample mini-batch
            batch_indices = np.random.choice(self.n_samples, batch_size, replace=False)

            # Compute stochastic gradient
            stoch_grad = self.stochastic_gradient(w, batch_indices)
            full_grad = self.full_gradient(w)

            history['grad_norm'].append(np.linalg.norm(full_grad))
            history['stoch_grad_norm'].append(np.linalg.norm(stoch_grad))

            w = w - lr * stoch_grad
            history['w'].append(w.copy())
            history['loss'].append(self.loss(w))

        return w, history


# Generate synthetic data
np.random.seed(42)
n_samples = 500
n_features = 2

# True parameters
w_true = np.array([2.0, -1.5])

# Generate data: y = Xw + noise
X = np.random.randn(n_samples, n_features)
y = X @ w_true + 0.5 * np.random.randn(n_samples)

# Create model
model = LinearRegressionSGD(X, y)

# Initial weights
w0 = np.array([0.0, 0.0])

print(f"Data shape: X={X.shape}, y={y.shape}")
print(f"True weights: {w_true}")
print(f"Initial loss: {model.loss(w0):.4f}")


In [ ]:
# Compare GD vs SGD
w_gd, hist_gd = model.gd(w0, lr=0.1, max_iter=100)
w_sgd, hist_sgd = model.sgd(w0, lr=0.1, max_iter=100, batch_size=1)
w_minibatch, hist_mb = model.sgd(w0, lr=0.1, max_iter=100, batch_size=32)

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Loss curves
ax = axes[0, 0]
ax.semilogy(hist_gd['loss'], 'b-', linewidth=2, label='Full-batch GD')
ax.semilogy(hist_sgd['loss'], 'r-', alpha=0.7, label='SGD (batch=1)')
ax.semilogy(hist_mb['loss'], 'g-', alpha=0.7, label='Mini-batch SGD (batch=32)')
ax.axhline(y=model.loss(w_true), color='black', linestyle='--', label='Optimal loss')
ax.set_xlabel('Iteration')
ax.set_ylabel('Loss (log scale)')
ax.set_title('Loss Convergence: GD vs SGD')
ax.legend()
ax.grid(True, alpha=0.3)

# Gradient norms
ax = axes[0, 1]
ax.semilogy(hist_gd['grad_norm'], 'b-', linewidth=2, label='GD gradient norm')
ax.semilogy(hist_sgd['stoch_grad_norm'], 'r-', alpha=0.5, label='SGD stochastic gradient norm')
ax.semilogy(hist_mb['stoch_grad_norm'], 'g-', alpha=0.5, label='Mini-batch stochastic gradient norm')
ax.set_xlabel('Iteration')
ax.set_ylabel('Gradient Norm (log scale)')
ax.set_title('Gradient Magnitude Over Time')
ax.legend()
ax.grid(True, alpha=0.3)

# Weight trajectory in 2D
ax = axes[1, 0]
w_hist_gd = np.array(hist_gd['w'])
w_hist_sgd = np.array(hist_sgd['w'])
w_hist_mb = np.array(hist_mb['w'])

ax.plot(w_hist_gd[:, 0], w_hist_gd[:, 1], 'b.-', linewidth=2, markersize=4, label='GD')
ax.plot(w_hist_sgd[:, 0], w_hist_sgd[:, 1], 'r.-', alpha=0.5, markersize=2, label='SGD')
ax.plot(w_hist_mb[:, 0], w_hist_mb[:, 1], 'g.-', alpha=0.5, markersize=2, label='Mini-batch')
ax.scatter([w_true[0]], [w_true[1]], color='black', s=200, marker='*', zorder=5, label='True w*')
ax.scatter([w0[0]], [w0[1]], color='purple', s=150, marker='s', zorder=5, label='Start')
ax.set_xlabel('w[0]')
ax.set_ylabel('w[1]')
ax.set_title('Weight Trajectory in Parameter Space')
ax.legend()
ax.grid(True, alpha=0.3)

# Final comparison
ax = axes[1, 1]
methods = ['GD', 'SGD (b=1)', 'Mini-batch (b=32)']
final_losses = [hist_gd['loss'][-1], hist_sgd['loss'][-1], hist_mb['loss'][-1]]
colors = ['blue', 'red', 'green']
bars = ax.bar(methods, final_losses, color=colors, alpha=0.7)
ax.axhline(y=model.loss(w_true), color='black', linestyle='--', label='Optimal')
ax.set_ylabel('Final Loss')
ax.set_title('Final Loss Comparison')
ax.legend()

# Add value labels
for bar, val in zip(bars, final_losses):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
            f'{val:.4f}', ha='center', va='bottom')

plt.tight_layout()
plt.show()

print(f"\nFinal Results:")
print(f"  True weights: {w_true}")
print(f"  GD weights: {w_gd}, loss={hist_gd['loss'][-1]:.6f}")
print(f"  SGD weights: {w_sgd}, loss={hist_sgd['loss'][-1]:.6f}")
print(f"  Mini-batch weights: {w_minibatch}, loss={hist_mb['loss'][-1]:.6f}")


## 2. Mini-Batch Size Effects

The batch size $B$ affects:
- **Variance**: Larger batches have lower variance gradients
- **Computation**: Larger batches can be parallelized better on GPUs
- **Generalization**: Smaller batches may generalize better (implicit regularization)
- **Convergence speed**: Trade-off between noise and progress per iteration


In [ ]:
def analyze_batch_size_effects():
    """Analyze how batch size affects SGD behavior."""

    batch_sizes = [1, 4, 16, 64, 128, n_samples]  # Last one is full batch
    colors = plt.cm.viridis(np.linspace(0, 1, len(batch_sizes)))

    fig, axes = plt.subplots(2, 2, figsize=(14, 10))

    results = {}

    for batch_size, color in zip(batch_sizes, colors):
        label = f'B={batch_size}' if batch_size < n_samples else 'Full batch'

        # Run multiple trials to measure variance
        n_trials = 5
        all_losses = []

        for trial in range(n_trials):
            np.random.seed(trial * 100)
            _, hist = model.sgd(w0, lr=0.05, max_iter=200, batch_size=batch_size)
            all_losses.append(hist['loss'])

        all_losses = np.array(all_losses)
        mean_loss = np.mean(all_losses, axis=0)
        std_loss = np.std(all_losses, axis=0)

        results[batch_size] = {'mean': mean_loss, 'std': std_loss}

        # Plot mean with uncertainty
        axes[0, 0].semilogy(mean_loss, color=color, linewidth=2, label=label)
        axes[0, 0].fill_between(range(len(mean_loss)),
                                 np.maximum(mean_loss - std_loss, 1e-10),
                                 mean_loss + std_loss,
                                 color=color, alpha=0.2)

    axes[0, 0].set_xlabel('Iteration')
    axes[0, 0].set_ylabel('Loss (log scale)')
    axes[0, 0].set_title('Loss vs Batch Size (with variance bands)')
    axes[0, 0].legend()
    axes[0, 0].grid(True, alpha=0.3)

    # Gradient variance for different batch sizes
    ax = axes[0, 1]

    batch_sizes_to_analyze = [1, 4, 16, 64]
    grad_variances = []

    for bs in batch_sizes_to_analyze:
        # Compute gradient variance at a fixed point
        w_test = np.array([1.0, -0.5])
        grads = []
        for _ in range(100):
            batch_idx = np.random.choice(n_samples, bs, replace=False)
            g = model.stochastic_gradient(w_test, batch_idx)
            grads.append(g)
        grads = np.array(grads)
        variance = np.mean(np.var(grads, axis=0))
        grad_variances.append(variance)

    ax.loglog(batch_sizes_to_analyze, grad_variances, 'bo-', linewidth=2, markersize=10)
    ax.loglog(batch_sizes_to_analyze, [grad_variances[0]/bs for bs in batch_sizes_to_analyze],
              'r--', linewidth=2, label='O(1/B) theoretical')
    ax.set_xlabel('Batch Size')
    ax.set_ylabel('Gradient Variance')
    ax.set_title('Gradient Variance vs Batch Size\n(Variance decreases as O(1/B))')
    ax.legend()
    ax.grid(True, alpha=0.3)

    # Iterations vs Batch Size for fixed accuracy
    ax = axes[1, 0]
    target_loss = model.loss(w_true) * 1.1  # 10% above optimal

    iters_to_target = []
    for batch_size in batch_sizes[:-1]:
        np.random.seed(42)
        _, hist = model.sgd(w0, lr=0.05, max_iter=1000, batch_size=batch_size)

        # Find first iteration below target
        for i, loss in enumerate(hist['loss']):
            if loss < target_loss:
                iters_to_target.append(i)
                break
        else:
            iters_to_target.append(1000)

    ax.semilogx(batch_sizes[:-1], iters_to_target, 'go-', linewidth=2, markersize=10)
    ax.set_xlabel('Batch Size')
    ax.set_ylabel('Iterations to Target Loss')
    ax.set_title(f'Iterations to Reach Loss < {target_loss:.4f}')
    ax.grid(True, alpha=0.3)

    # Wall-clock time simulation
    ax = axes[1, 1]

    # Assume: time per iteration = base_time + batch_size * time_per_sample
    # With GPU parallelization, time per iteration is roughly constant for small batches
    base_time = 1.0
    time_per_sample = 0.01

    wall_times = []
    for i, (bs, iters) in enumerate(zip(batch_sizes[:-1], iters_to_target)):
        # Simplified model: time = iters * (base + min(bs, 32) * time_per_sample)
        # Assumes GPU can parallelize up to 32 samples
        effective_time = iters * (base_time + min(bs, 32) * time_per_sample)
        wall_times.append(effective_time)

    ax.semilogx(batch_sizes[:-1], wall_times, 'mo-', linewidth=2, markersize=10)
    ax.set_xlabel('Batch Size')
    ax.set_ylabel('Simulated Wall-Clock Time')
    ax.set_title('Total Time to Convergence\n(Simplified GPU model)')
    ax.grid(True, alpha=0.3)

    plt.tight_layout()
    plt.show()

analyze_batch_size_effects()


## 3. Variance Reduction Animation

SGD gradients are noisy estimates of the true gradient. **Variance reduction** techniques aim to reduce this noise:

- **SVRG (Stochastic Variance Reduced Gradient)**: Periodically compute full gradient
- **SAG/SAGA**: Maintain gradient table
- **Increasing batch size**: Simple but effective

Let's visualize the noisy vs smooth gradient directions.


In [ ]:
def visualize_gradient_variance():
    """Visualize gradient noise at different batch sizes."""

    fig, axes = plt.subplots(2, 3, figsize=(15, 10))

    # Test point
    w_test = np.array([0.5, 0.5])

    batch_sizes = [1, 4, 16, 64, 128, n_samples]

    for ax, bs in zip(axes.flat, batch_sizes):
        # Compute many stochastic gradients
        n_samples_grad = 50
        grads = []

        for _ in range(n_samples_grad):
            batch_idx = np.random.choice(n_samples, min(bs, n_samples), replace=False)
            g = model.stochastic_gradient(w_test, batch_idx)
            grads.append(g)

        grads = np.array(grads)
        true_grad = model.full_gradient(w_test)

        # Plot gradient vectors
        for g in grads:
            ax.arrow(0, 0, g[0]*0.3, g[1]*0.3, head_width=0.02,
                    color='blue', alpha=0.3)

        # True gradient in red
        ax.arrow(0, 0, true_grad[0]*0.3, true_grad[1]*0.3, head_width=0.05,
                color='red', linewidth=2, label='True gradient')

        # Mean of stochastic gradients
        mean_grad = np.mean(grads, axis=0)
        ax.arrow(0, 0, mean_grad[0]*0.3, mean_grad[1]*0.3, head_width=0.05,
                color='green', linewidth=2, label='Mean stochastic')

        variance = np.mean(np.var(grads, axis=0))
        title = f'Batch size = {bs}' if bs < n_samples else 'Full batch'
        ax.set_title(f'{title}\nVariance: {variance:.4f}')
        ax.set_xlim(-0.8, 0.8)
        ax.set_ylim(-0.8, 0.8)
        ax.set_aspect('equal')
        ax.axhline(y=0, color='gray', linestyle='-', alpha=0.3)
        ax.axvline(x=0, color='gray', linestyle='-', alpha=0.3)
        ax.grid(True, alpha=0.3)

        if bs == 1:
            ax.legend(loc='upper right', fontsize=8)

    plt.suptitle('Gradient Variance Visualization\n(Blue: stochastic gradients, Red: true gradient, Green: mean)',
                 fontsize=14)
    plt.tight_layout()
    plt.show()

visualize_gradient_variance()


In [ ]:
def create_sgd_animation():
    """Create animation showing SGD vs GD optimization paths."""

    # Create a simple 2D objective for visualization
    # f(x,y) = x^2 + 10*y^2 (elongated bowl)

    def f(x, y):
        return x**2 + 10*y**2

    def grad_f(x, y):
        return np.array([2*x, 20*y])

    def noisy_grad_f(x, y, noise_scale=1.0):
        true_grad = grad_f(x, y)
        noise = np.random.randn(2) * noise_scale
        return true_grad + noise

    # Generate trajectories
    np.random.seed(42)
    x0, y0 = 4.0, 2.0
    lr = 0.05
    n_steps = 50

    # GD trajectory
    gd_path = [(x0, y0)]
    x, y = x0, y0
    for _ in range(n_steps):
        g = grad_f(x, y)
        x, y = x - lr * g[0], y - lr * g[1]
        gd_path.append((x, y))
    gd_path = np.array(gd_path)

    # SGD trajectory (high noise)
    sgd_path = [(x0, y0)]
    x, y = x0, y0
    for _ in range(n_steps):
        g = noisy_grad_f(x, y, noise_scale=3.0)
        x, y = x - lr * g[0], y - lr * g[1]
        sgd_path.append((x, y))
    sgd_path = np.array(sgd_path)

    # Create figure
    fig, ax = plt.subplots(figsize=(10, 8))

    # Plot contours
    xx = np.linspace(-5, 5, 100)
    yy = np.linspace(-3, 3, 100)
    XX, YY = np.meshgrid(xx, yy)
    ZZ = f(XX, YY)

    levels = [0.5, 1, 2, 5, 10, 20, 40]
    cs = ax.contour(XX, YY, ZZ, levels=levels, cmap='Blues')
    ax.clabel(cs, inline=True, fontsize=8)

    # Plot full trajectories (faded)
    ax.plot(gd_path[:, 0], gd_path[:, 1], 'b-', alpha=0.3, linewidth=1)
    ax.plot(sgd_path[:, 0], sgd_path[:, 1], 'r-', alpha=0.3, linewidth=1)

    # Current positions (will be updated)
    gd_point, = ax.plot([], [], 'bo', markersize=12, label='GD')
    sgd_point, = ax.plot([], [], 'ro', markersize=12, label='SGD')

    # Trails
    gd_trail, = ax.plot([], [], 'b-', linewidth=2)
    sgd_trail, = ax.plot([], [], 'r-', linewidth=2)

    ax.plot([x0], [y0], 'gs', markersize=15, label='Start')
    ax.plot([0], [0], 'k*', markersize=20, label='Optimum')

    ax.set_xlabel('x')
    ax.set_ylabel('y')
    ax.set_title('GD vs SGD Optimization')
    ax.legend(loc='upper right')
    ax.set_xlim(-5, 5)
    ax.set_ylim(-3, 3)
    ax.grid(True, alpha=0.3)

    def init():
        gd_point.set_data([], [])
        sgd_point.set_data([], [])
        gd_trail.set_data([], [])
        sgd_trail.set_data([], [])
        return gd_point, sgd_point, gd_trail, sgd_trail

    def animate(i):
        gd_point.set_data([gd_path[i, 0]], [gd_path[i, 1]])
        sgd_point.set_data([sgd_path[i, 0]], [sgd_path[i, 1]])
        gd_trail.set_data(gd_path[:i+1, 0], gd_path[:i+1, 1])
        sgd_trail.set_data(sgd_path[:i+1, 0], sgd_path[:i+1, 1])
        ax.set_title(f'GD vs SGD Optimization - Step {i}\nGD loss: {f(gd_path[i,0], gd_path[i,1]):.4f}, SGD loss: {f(sgd_path[i,0], sgd_path[i,1]):.4f}')
        return gd_point, sgd_point, gd_trail, sgd_trail

    anim = animation.FuncAnimation(fig, animate, init_func=init,
                                   frames=n_steps+1, interval=100, blit=True)
    plt.close()
    return anim

# Create and display animation
anim = create_sgd_animation()
HTML(anim.to_jshtml())


## 4. Learning Rate Scheduling

Learning rate schedules help balance exploration (high LR) and convergence (low LR):

| Schedule | Formula | Properties |
|----------|---------|------------|
| Constant | $\alpha_t = \alpha_0$ | Simple, may not converge |
| Step decay | $\alpha_t = \alpha_0 \cdot \gamma^{\lfloor t/s \rfloor}$ | Easy to tune |
| Exponential | $\alpha_t = \alpha_0 \cdot e^{-\lambda t}$ | Smooth decay |
| Inverse | $\alpha_t = \alpha_0 / (1 + \lambda t)$ | Theory-motivated |
| Cosine | $\alpha_t = \alpha_{min} + \frac{1}{2}(\alpha_0 - \alpha_{min})(1 + \cos(\pi t / T))$ | Popular in deep learning |


In [ ]:
def learning_rate_schedules():
    """Compare different learning rate schedules."""

    # Define schedules
    def constant(t, lr0=0.1, **kwargs):
        return lr0

    def step_decay(t, lr0=0.1, gamma=0.5, step_size=50, **kwargs):
        return lr0 * (gamma ** (t // step_size))

    def exponential(t, lr0=0.1, decay=0.02, **kwargs):
        return lr0 * np.exp(-decay * t)

    def inverse(t, lr0=0.1, decay=0.01, **kwargs):
        return lr0 / (1 + decay * t)

    def cosine(t, lr0=0.1, lr_min=0.001, T=200, **kwargs):
        return lr_min + 0.5 * (lr0 - lr_min) * (1 + np.cos(np.pi * t / T))

    def warmup_cosine(t, lr0=0.1, lr_min=0.001, T=200, warmup=20, **kwargs):
        if t < warmup:
            return lr0 * t / warmup
        else:
            t_adj = t - warmup
            T_adj = T - warmup
            return lr_min + 0.5 * (lr0 - lr_min) * (1 + np.cos(np.pi * t_adj / T_adj))

    schedules = {
        'Constant': constant,
        'Step Decay': step_decay,
        'Exponential': exponential,
        'Inverse (1/t)': inverse,
        'Cosine': cosine,
        'Warmup + Cosine': warmup_cosine,
    }

    T = 200
    t_range = np.arange(T)

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    # Plot schedules
    ax = axes[0]
    for name, schedule in schedules.items():
        lrs = [schedule(t) for t in t_range]
        ax.plot(t_range, lrs, linewidth=2, label=name)

    ax.set_xlabel('Iteration')
    ax.set_ylabel('Learning Rate')
    ax.set_title('Learning Rate Schedules')
    ax.legend()
    ax.grid(True, alpha=0.3)
    ax.set_ylim(0, 0.12)

    # Compare convergence with different schedules
    ax = axes[1]

    colors = plt.cm.tab10(np.linspace(0, 1, len(schedules)))

    for (name, schedule), color in zip(schedules.items(), colors):
        # Run SGD with this schedule
        np.random.seed(42)
        w = w0.copy()
        losses = [model.loss(w)]

        for t in range(200):
            lr = schedule(t)
            batch_idx = np.random.choice(n_samples, 32, replace=False)
            grad = model.stochastic_gradient(w, batch_idx)
            w = w - lr * grad
            losses.append(model.loss(w))

        ax.semilogy(losses, color=color, linewidth=2, label=name)

    ax.axhline(y=model.loss(w_true), color='black', linestyle='--', alpha=0.5, label='Optimal')
    ax.set_xlabel('Iteration')
    ax.set_ylabel('Loss (log scale)')
    ax.set_title('Convergence with Different LR Schedules')
    ax.legend()
    ax.grid(True, alpha=0.3)

    plt.tight_layout()
    plt.show()

learning_rate_schedules()


In [ ]:
def interactive_lr_schedule(schedule='cosine', lr0=0.1, decay=0.02, warmup=20):
    """Interactive exploration of learning rate schedules."""

    T = 200
    t_range = np.arange(T)

    # Define schedule based on selection
    if schedule == 'constant':
        lrs = [lr0] * T
    elif schedule == 'exponential':
        lrs = [lr0 * np.exp(-decay * t) for t in t_range]
    elif schedule == 'cosine':
        lrs = [0.001 + 0.5 * (lr0 - 0.001) * (1 + np.cos(np.pi * t / T)) for t in t_range]
    elif schedule == 'warmup_cosine':
        def warmup_cosine_fn(t):
            if t < warmup:
                return lr0 * t / warmup
            else:
                t_adj = t - warmup
                T_adj = T - warmup
                return 0.001 + 0.5 * (lr0 - 0.001) * (1 + np.cos(np.pi * t_adj / T_adj))
        lrs = [warmup_cosine_fn(t) for t in t_range]
    elif schedule == 'step':
        lrs = [lr0 * (0.5 ** (t // 50)) for t in t_range]

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    # Schedule plot
    ax = axes[0]
    ax.plot(t_range, lrs, 'b-', linewidth=2)
    ax.fill_between(t_range, 0, lrs, alpha=0.3)
    ax.set_xlabel('Iteration')
    ax.set_ylabel('Learning Rate')
    ax.set_title(f'Learning Rate Schedule: {schedule}')
    ax.grid(True, alpha=0.3)

    # Run SGD with this schedule
    np.random.seed(42)
    w = w0.copy()
    losses = [model.loss(w)]

    for t in range(T):
        lr = lrs[t]
        batch_idx = np.random.choice(n_samples, 32, replace=False)
        grad = model.stochastic_gradient(w, batch_idx)
        w = w - lr * grad
        losses.append(model.loss(w))

    ax = axes[1]
    ax.semilogy(losses, 'b-', linewidth=2)
    ax.axhline(y=model.loss(w_true), color='red', linestyle='--', label='Optimal')
    ax.set_xlabel('Iteration')
    ax.set_ylabel('Loss (log scale)')
    ax.set_title(f'Convergence (final loss: {losses[-1]:.6f})')
    ax.legend()
    ax.grid(True, alpha=0.3)

    plt.tight_layout()
    plt.show()

if WIDGETS_AVAILABLE:
    interact(interactive_lr_schedule,
             schedule=Dropdown(options=['constant', 'exponential', 'cosine', 'warmup_cosine', 'step'],
                               value='cosine', description='Schedule'),
             lr0=FloatSlider(min=0.01, max=0.5, step=0.01, value=0.1, description='Initial LR'),
             decay=FloatSlider(min=0.001, max=0.1, step=0.001, value=0.02, description='Decay'),
             warmup=IntSlider(min=0, max=50, step=5, value=20, description='Warmup'))
else:
    for sched in ['constant', 'cosine', 'warmup_cosine']:
        interactive_lr_schedule(schedule=sched)


---

## Interview Questions

### Question 1: SGD vs Full-Batch GD
**Q: Why does SGD often generalize better than full-batch gradient descent, even though it uses noisier gradients?**

<details>
<summary>Click for Answer</summary>

**Key Points:**

1. **Implicit Regularization**: The noise in SGD acts as implicit regularization, preventing the model from overfitting to the training data too precisely.

2. **Escaping Sharp Minima**: SGD's noise helps escape sharp local minima that may not generalize well, favoring flatter minima that tend to generalize better.

3. **Exploration**: The stochastic nature allows SGD to explore the loss landscape more broadly rather than deterministically following a single path.

4. **Minibatch Diversity**: Each update sees different data, preventing memorization of the training set order.

5. **Theoretical Support**: Recent research (e.g., "Sharp Minima Can Generalize For Deep Nets") shows that the noise scale of SGD (learning rate / batch size) affects the width of minima found.

**Practical Implications:**
- Larger batch sizes may require different learning rates
- Very large batches can hurt generalization
- The "generalization gap" often increases with batch size
</details>

### Question 2: Learning Rate and Batch Size
**Q: When you increase the batch size, how should you adjust the learning rate? Why?**

<details>
<summary>Click for Answer</summary>

**Linear Scaling Rule**: When multiplying batch size by $k$, multiply learning rate by $k$.

**Intuition:**
- Larger batches give lower variance gradient estimates
- To maintain similar "effective noise" in the optimization, increase step size
- The gradient variance decreases as $O(1/B)$ where $B$ is batch size
- To cover same distance in parameter space over an epoch, need larger steps with fewer updates

**Mathematical Justification:**
- SGD update: $w_{t+1} = w_t - \alpha \cdot g_t$
- Expected progress per epoch with batch $B$: $\sim (n/B) \cdot \alpha \cdot \nabla f$
- Doubling $B$ halves updates per epoch, so double $\alpha$ to maintain progress

**Caveats:**
- Linear scaling breaks down for very large batches
- May need "warmup" period when using large learning rates
- Different optimizers (Adam, etc.) may have different scaling rules
</details>

### Question 3: Variance Reduction
**Q: Explain the key idea behind SVRG (Stochastic Variance Reduced Gradient). When would you use it over vanilla SGD?**

<details>
<summary>Click for Answer</summary>

**SVRG Key Idea:**
1. Periodically compute full gradient $\tilde{g} = \nabla f(\tilde{w})$ at a "snapshot" point $\tilde{w}$
2. For each stochastic update, use the variance-reduced gradient:
   $$g_t = \nabla f_i(w_t) - \nabla f_i(\tilde{w}) + \tilde{g}$$
3. This corrects the stochastic gradient using the difference between local and global information

**Why It Works:**
- Near the optimum, $\nabla f_i(w_t) \approx \nabla f_i(\tilde{w})$, so variance vanishes
- Achieves linear convergence rate for strongly convex functions (vs. sublinear for SGD)
- Variance goes to zero as we approach optimum

**When to Use SVRG over SGD:**
- Strongly convex problems (logistic regression, ridge regression)
- When high precision is needed
- Moderate dataset sizes where periodic full-gradient passes are acceptable
- When SGD's convergence stalls due to variance

**When NOT to Use:**
- Very large datasets (full gradient too expensive)
- Deep learning (non-convex, SGD's noise may help generalization)
- Online learning (no fixed dataset)
</details>

---


## Exercises

### Exercise 1: Momentum SGD
Implement SGD with momentum and compare with vanilla SGD:
$$v_{t+1} = \beta v_t + g_t$$
$$w_{t+1} = w_t - \alpha v_{t+1}$$


In [ ]:
# Exercise 1: Your solution here

def sgd_momentum(model, w0, lr=0.01, beta=0.9, max_iter=200, batch_size=32):
    """
    SGD with momentum.

    TODO: Implement this function

    Parameters:
    -----------
    beta : float
        Momentum coefficient

    Returns:
    --------
    w : final weights
    history : dict with 'loss' key
    """
    pass

# Compare vanilla SGD vs Momentum SGD
# Hint: Momentum should converge faster and smoother


### Exercise 2: Adaptive Learning Rate
Implement AdaGrad: an adaptive learning rate method that adjusts per-parameter learning rates based on historical gradients.

$$G_t = G_{t-1} + g_t \odot g_t$$
$$w_{t+1} = w_t - \frac{\alpha}{\sqrt{G_t + \epsilon}} \odot g_t$$


In [ ]:
# Exercise 2: Your solution here

def adagrad(model, w0, lr=0.5, max_iter=200, batch_size=32, eps=1e-8):
    """
    AdaGrad optimizer.

    TODO: Implement this function

    Hint: Maintain sum of squared gradients G_t
    """
    pass

# Test AdaGrad and compare with vanilla SGD
# AdaGrad should adapt to the geometry of the loss surface


### Exercise 3: Convergence Analysis
Run SGD with different random seeds and analyze:
1. Variance in final loss across runs
2. Variance in number of iterations to reach a target loss
3. How does batch size affect these variances?

Plot your findings.


In [ ]:
# Exercise 3: Your solution here

def analyze_sgd_variance(model, w0, batch_sizes=[1, 8, 32, 128], n_trials=20, max_iter=200):
    """
    Analyze variance of SGD across multiple runs.

    TODO: Implement this analysis

    For each batch size:
    - Run n_trials experiments with different random seeds
    - Record final loss for each trial
    - Compute mean and std of final losses

    Plot: batch_size vs final_loss (with error bars)
    """
    pass

# Run analysis
# analyze_sgd_variance(model, w0)
